In [2]:
#!/usr/bin/env python3
"""
Export de toutes les stations de vélos en libre-service (VLS) en France.
Compatible Microsoft Fabric (Lakehouse Couche Bronze).

- Source : flux GBFS référencés sur transport.data.gouv.fr
- Garde UNIQUEMENT : vélos (form_factor = bicycle) avec stations physiques et capacité > 0
- Exclut : trottinettes, scooters, vélos flottants (free-floating), zones de dépôt virtuelles
"""
import csv
import json
import time
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
import requests

# ----------------------------------------------------------------------------
# Configuration Emplacement Microsoft Fabric Lakehouse (Bronze)
# ----------------------------------------------------------------------------
fabric_path = Path('/lakehouse/default/Files')

if fabric_path.is_dir():
    TARGET_DIR = fabric_path / 'bronze' / 'gbfs_velhop'
    print("Environnement Microsoft Fabric détecté : écriture dans Lakehouse/Files/bronze/gbfs_velhop")
else:
    # Secours local si exécuté hors Fabric
    TARGET_DIR = Path('bronze/gbfs_velhop')
    print("Mode local : écriture dans ./bronze/gbfs_velhop")

TARGET_DIR.mkdir(parents=True, exist_ok=True)

CATALOG = "https://transport.data.gouv.fr/api/datasets?type=vehicles-sharing"
HEADERS = {"User-Agent": "vls-stations-fr/2.0"}
TIMEOUT = 8
WORKERS = 16

# Opérateurs de véhicules flottants / trottinettes : exclus d'office
FLOTTANTS = ("dott", "lime", "voi", "tier", "bird", "bolt", "pony", "zeus",
             "cityscoot", "cooltra", "yego", "spin", "hellobike",
             "trottinette", "scooter")


# ----------------------------------------------------------------------------
# Utilitaires
# ----------------------------------------------------------------------------
def get_json(url, retries=2):
    last = None
    for i in range(retries):
        try:
            r = requests.get(url, headers=HEADERS, timeout=TIMEOUT)
            r.raise_for_status()
            return r.json()
        except Exception as e:
            last = e
            time.sleep(1.5 * (i + 1))
    raise last


def txt(v):
    if isinstance(v, list):
        return v[0].get("text", "") if v else ""
    return v or ""


def get_feeds(gbfs):
    data = gbfs.get("data", {})
    feeds = data.get("feeds")
    if feeds is None:
        for v in data.values():
            if isinstance(v, dict) and "feeds" in v:
                feeds = v["feeds"]
                break
    return {f["name"]: f["url"] for f in (feeds or [])}


NON_VELO = ("autopartage", "auto-partage", "voiture", "covoiturage", "citiz",
            "getaround", "free2move", "zity", "ubeeqo", "clem")


def est_service_velo(ds):
    titre = (ds.get("title") or "").lower()
    tags = [str(t).lower() for t in (ds.get("tags") or [])]
    if any(m in titre for m in NON_VELO):
        return False
    return titre.startswith("vls") or "vls" in tags or "velo" in titre or "vélo" in titre


def est_flottant(titre):
    t = (titre or "").lower()
    return any(m in t for m in FLOTTANTS)


def station_valide(s):
    if s.get("is_virtual_station"):
        return False
    if s.get("station_area"):
        return False
    if s.get("is_valet_station"):
        return False

    # Exclure capacité <= 0 ou manquante
    cap = s.get("capacity")
    try:
        if cap is None or int(cap) <= 0:
            return False
    except (TypeError, ValueError):
        return False

    # Exclure bornes de test / itinérantes
    nom = txt(s.get("name")).lower()
    if not nom or any(mot in nom for mot in ("borne itinérante", "borne cgb", "station test")):
        return False

    if s.get("lat") is None or s.get("lon") is None:
        return False
    try:
        lat, lon = float(s["lat"]), float(s["lon"])
    except (TypeError, ValueError):
        return False
    return not (lat == 0 and lon == 0)


# ----------------------------------------------------------------------------
# Traitement d'un réseau
# ----------------------------------------------------------------------------
def traiter(ds_title, gbfs_url):
    gbfs = get_json(gbfs_url)
    feeds = get_feeds(gbfs)

    if "station_information" not in feeds:
        return [], "exclu : pas de station_information (vélos flottants)"

    velo_ids = None
    if "vehicle_types" in feeds:
        try:
            vt = get_json(feeds["vehicle_types"])["data"]["vehicle_types"]
            velo_ids = {v["vehicle_type_id"] for v in vt if v.get("form_factor") == "bicycle"}
            if not velo_ids:
                return [], "exclu : aucun vélo (trottinettes/scooters seulement)"
        except Exception:
            velo_ids = None

    info = get_json(feeds["station_information"])["data"]["stations"]

    stations, nb_virtuelles = [], 0
    for s in info:
        if not station_valide(s):
            nb_virtuelles += 1
            continue

        cap_types = s.get("vehicle_type_capacity") or s.get("vehicle_types_capacity")
        if velo_ids is not None and cap_types:
            ids = set(cap_types.keys()) if isinstance(cap_types, dict) else {c.get("vehicle_type_id") for c in cap_types}
            if not (ids & velo_ids):
                continue

        stations.append({
            "reseau": ds_title,
            "station_id": s.get("station_id"),
            "nom": txt(s.get("name")),
            "lat": float(s["lat"]),
            "lon": float(s["lon"]),
            "capacite": int(s.get("capacity")),
            "adresse": s.get("address", ""),
            "source_gbfs": gbfs_url,
        })

    statut = f"ok : {len(stations)} stations"
    if nb_virtuelles:
        statut += f" ({nb_virtuelles} zones virtuelles/invalides ignorées)"
    return stations, statut


# ----------------------------------------------------------------------------
# Main
# ----------------------------------------------------------------------------
def main():
    print("Téléchargement du catalogue transport.data.gouv.fr ...")
    catalogue = get_json(CATALOG)

    taches, vus = [], set()
    rapport = []
    for ds in catalogue:
        titre = ds.get("title")
        for res in ds.get("resources", []):
            if (res.get("format") or "").lower() != "gbfs":
                continue
            url = res.get("url")
            if not url or url in vus:
                continue
            vus.add(url)
            if not est_service_velo(ds):
                rapport.append((titre, url, "exclu : pas un service de vélos (ex. autopartage)"))
                continue
            if est_flottant(titre):
                rapport.append((titre, url, "exclu : opérateur flottant/trottinettes"))
                continue
            taches.append((titre, url))

    print(f"{len(taches)} flux GBFS à analyser ...", flush=True)

    toutes = []
    with ThreadPoolExecutor(max_workers=WORKERS) as ex:
        futs = {ex.submit(traiter, t, u): (t, u) for t, u in taches}
        for i, fut in enumerate(as_completed(futs), 1):
            t, u = futs[fut]
            try:
                st, statut = fut.result()
                toutes.extend(st)
            except Exception as e:
                statut = f"erreur : {e}"
            rapport.append((t, u, statut))
            if i % 5 == 0 or i == len(taches):
                print(f"  {i}/{len(taches)} flux traités...", flush=True)

    # Dédoublonnage
    vues, finales = set(), []
    for s in toutes:
        cle = (round(s["lat"], 5), round(s["lon"], 5), s["nom"].strip().lower())
        if cle in vues:
            continue
        vues.add(cle)
        finales.append(s)

    if not finales:
        print("Aucune station récupérée, regarde rapport_reseaux.csv")
    else:
        champs = list(finales[0].keys())
        csv_file = TARGET_DIR / "stations_velos_france.csv"
        with open(csv_file, "w", newline="", encoding="utf-8-sig") as f:
            w = csv.DictWriter(f, fieldnames=champs, delimiter=";")
            w.writeheader()
            w.writerows(finales)

        geojson = {
            "type": "FeatureCollection",
            "features": [{
                "type": "Feature",
                "geometry": {"type": "Point", "coordinates": [s["lon"], s["lat"]]},
                "properties": {k: v for k, v in s.items() if k not in ("lat", "lon")},
            } for s in finales],
        }
        geojson_file = TARGET_DIR / "stations_velos_france.geojson"
        with open(geojson_file, "w", encoding="utf-8") as f:
            json.dump(geojson, f, ensure_ascii=False)

    rapport_file = TARGET_DIR / "rapport_reseaux.csv"
    with open(rapport_file, "w", newline="", encoding="utf-8-sig") as f:
        w = csv.writer(f, delimiter=";")
        w.writerow(["reseau", "url_gbfs", "statut"])
        w.writerows(sorted(rapport, key=lambda r: str(r[0])))

    nb_reseaux = len({s["reseau"] for s in finales})
    nb_err = sum(1 for r in rapport if r[2].startswith("erreur"))
    print(f"\nTerminé : {len(finales)} stations, {nb_reseaux} réseaux, {nb_err} flux en erreur.")
    print(f"Fichiers enregistrés dans : {TARGET_DIR}")


if __name__ == "__main__":
    main()
